#### Importing Libraries

In [59]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

import joblib

#### Loading the datasets

In [60]:
train_path = "../Data/raw/UNSW_NB15_training-set.csv"
test_path = "../Data/raw/UNSW_NB15_testing-set.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

Dropping unnecessary columns

In [61]:
df_train = df_train.drop(columns = ['id','attack_cat'])
df_test = df_test.drop(columns = ['id','attack_cat'])

Separating X and y

In [62]:
X_train = df_train.drop('label',axis=1)
y_train = df_train['label']

X_test = df_test.drop('label',axis=1)
y_test = df_test['label']

print(X_train.shape)
print(y_train.shape)

print(X_test.shape)
print(y_test.shape)

(175341, 42)
(175341,)
(82332, 42)
(82332,)


Identifying categorical columns

In [63]:
categorical_cols = X_train.select_dtypes(include="str").columns

print(categorical_cols)

Index(['proto', 'service', 'state'], dtype='str')


Identifying numerical columns

In [64]:
numerical_cols = X_train.select_dtypes(include=np.number).columns

print(numerical_cols)

Index(['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl',
       'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit',
       'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean',
       'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src',
       'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm',
       'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd',
       'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports'],
      dtype='str')


Creating preprocessing pipeline

In [65]:
preprocessor = ColumnTransformer(transformers=[
        ("num", StandardScaler(),numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"),categorical_cols)])

In [66]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)


(175341, 194)
(82332, 194)


Saving feature names during preprocessing

In [67]:
feature_names = preprocessor.get_feature_names_out()
print(len(feature_names))

np.save("../data/processed/feature_names.npy",feature_names)

194


Saving the preprocessor

In [68]:
joblib.dump(preprocessor,"../models/preprocessor.pkl")

['../models/preprocessor.pkl']

Saving processed features

In [69]:
X_train_processed_df = pd.DataFrame(X_train_processed.toarray())
X_test_processed_df = pd.DataFrame(X_test_processed.toarray())

X_train_processed_df.to_csv("../data/processed/X_train_processed.csv",index=False)
X_test_processed_df.to_csv("../data/processed/X_test_processed.csv",index=False)

In [70]:
y_train.to_csv("../data/processed/y_train.csv",index=False)
y_test.to_csv("../data/processed/y_test.csv",index=False)

## Preprocessing Summary

The following preprocessing steps were completed:

- Removed `id` and `attack_cat` columns.
- Separated features (`X`) and target (`y`).
- Identified numerical and categorical features.
- Applied `StandardScaler` to numerical features.
- Applied `OneHotEncoder` to categorical features using a `ColumnTransformer`.
- Saved the fitted preprocessor as `preprocessor.pkl`.
- Saved processed training and testing datasets for future model training.